In [1]:
import pandas as pd
import numpy as np
from src import run_experiment, Config
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer, KNNImputer
from matplotlib import pyplot as plt
from collections import Counter


In [2]:
df = pd.read_csv('../data/app_train_nan90.csv')

In [ ]:
# target = df['TARGET']

In [ ]:
# Солвер 'saga' с penalty='elasticnet' очень долго сходится, поэтому сделаем пока чистую L2-регуляризацию
def log_reg_param_space(trial):
    return {
        "C": trial.suggest_float('C', 1e-3, 1e2, log=True),
        # "l1_ratio": trial.suggest_float('l1_ratio', 0, 1, step=0.1),
        # "penalty": trial.suggest_categorical('penalty', ['elasticnet']),
        # "solver": trial.suggest_categorical('solver', ['saga']),
        "max_iter": trial.suggest_int('max_iter', 100, 1000, step=100)
    }

# exp_name_lr = f'90_nan'

# pipeline_lr, results = run_experiment(
#     exp_name_lr,
#     LogisticRegression,
#     log_reg_param_space,
#     df,
#     n_trials=1,
#     cv_num=5,
#     save_model=True
# )

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def decision_tree_param_space(trial):
    return {
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        "max_depth": trial.suggest_int("max_depth", 1, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

exp_name_dt = f'90_nan'

# run_experiment(
#     exp_name_dt,
#     DecisionTreeClassifier,
#     decision_tree_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=True
# )

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def random_forest_param_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 1, 13),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False])
    }

exp_name_rf = f'90_nan'

# run_experiment(
#     exp_name_rf,
#     RandomForestClassifier,
#     random_forest_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=True
# )

In [3]:
from catboost import CatBoostClassifier, cv, Pool

def catboost_param_space(trial):
    return {
        "iterations": trial.suggest_int("iterations", 100, 600),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 0, 1),
        
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        
        "bootstrap_type": "Bernoulli",
        "subsample": trial.suggest_float("subsample", 0.5, 1),
        
        "verbose": False,
        "random_state": Config.SEED,

        "od_type": 'Iter',
        "od_wait": 30
        # Golden features
        # --per-float-feature-quantization 0:border_count=1024;1:border_count=1024
    }

# feature_importances_ shap

# exp_name_cb = f'90_nan'

# catboost_pipeline, results = run_experiment(
#     exp_name_cb,
#     CatBoostClassifier,
#     catboost_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=False
# )

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

df_imputed = pd.concat([df[num_cols], df[cat_cols].fillna('NA')], axis=1)
target = df_imputed['TARGET']

if 'TARGET' in df_imputed.columns:
    df_imputed.drop(columns=['TARGET'], inplace=True)

pool = Pool(df_imputed, label=target, cat_features=cat_cols)

In [ ]:
# Сразу же удалим столбцы, в которых слишком большой процент пропусков
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 1]

In [ ]:
cols_to_drop = missing_pct.index[missing_pct > 10].tolist()
df = df.drop(columns=cols_to_drop)

In [5]:
cv_results = cv(
    pool=pool,
    params={
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.1,
        "l2_leaf_reg": 3,
        "border_count": 32,
        "random_strength": 0.1,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        "bootstrap_type": "Bernoulli",
        "subsample": 0.8,
        "verbose": True,
        "random_state": Config.SEED,
        "od_type": 'Iter',
        "od_wait": 30
    },
    nfold=5,
    stratified=True,
    shuffle=True,
    plot=True
    
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Training on fold [0/5]
0:	test: 0.6997972	best: 0.6997972 (0)	total: 105ms	remaining: 1m 23s
1:	test: 0.7086509	best: 0.7086509 (1)	total: 204ms	remaining: 1m 21s
2:	test: 0.7162600	best: 0.7162600 (2)	total: 332ms	remaining: 1m 28s
3:	test: 0.7196122	best: 0.7196122 (3)	total: 435ms	remaining: 1m 26s
4:	test: 0.7220888	best: 0.7220888 (4)	total: 588ms	remaining: 1m 33s
5:	test: 0.7225159	best: 0.7225159 (5)	total: 711ms	remaining: 1m 34s
6:	test: 0.7254172	best: 0.7254172 (6)	total: 846ms	remaining: 1m 35s
7:	test: 0.7260334	best: 0.7260334 (7)	total: 951ms	remaining: 1m 34s
8:	test: 0.7291056	best: 0.7291056 (8)	total: 1.06s	remaining: 1m 33s
9:	test: 0.7310870	best: 0.7310870 (9)	total: 1.23s	remaining: 1m 37s
10:	test: 0.7335035	best: 0.7335035 (10)	total: 1.34s	remaining: 1m 36s
11:	test: 0.7344788	best: 0.7344788 (11)	total: 1.48s	remaining: 1m 37s
12:	test: 0.7361550	best: 0.7361550 (12)	total: 1.6s	remaining: 1m 36s
13:	test: 0.7380974	best: 0.7380974 (13)	total: 1.73s	remainin

KeyboardInterrupt: 

In [ ]:
cv_results['test-AUC-mean'].max()

In [ ]:
from lightgbm import LGBMClassifier

def lightgbm_param_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        # "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "objective": "binary",
        "data_sample_strategy": "goss",
        # "early_stopping_round": 50, необходимо передать в fit, а не в конструктор модели
        "metric": "auc",
        "verbose": -1,
        "random_state": Config.SEED
        # можно добавить использование cat_features
    }

# exp_name_lgbm = f'90_nan'

# run_experiment(
#     exp_name_lgbm,
#     LGBMClassifier,
#     lightgbm_param_space,
#     df,
#     n_trials=1,
#     cv_num=5,
#     save_model=False
# )

In [ ]:
from xgboost import XGBClassifier

def xgboost_param_space(trial):
    return {
        "verbosity": 0,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1, 11]),
        "objective": "binary:logistic",
        "eval_metric": 'auc',
        "seed": Config.SEED,

        # "gamma"
        # "max_cat_to_onehot"
        # "max_cat_threshold"
        # можно добавить использование cat_features
    }

# mask = target == 0
# sum_pos = np.sum(target, axis=0)
# sum_neg = np.sum(mask)
# print(sum_pos)
# print(sum_neg)
# scale_pos_weight = sum_neg / sum_pos
# print(scale_pos_weight) = 11

exp_name_xgb = f'90_nan'

# CMA-ES с Warm-Starting 
# Meta-Learn TPE — Лучший выбор для TPE

run_experiment(
    exp_name_xgb,
    XGBClassifier,
    xgboost_param_space,
    df,
    n_trials=10,
    cv_num=5,
    save_model=False
)

# Export to the PROD

Из всех моделей осталось выбрать наилучшую по Kaggle-score и дальше через FastAPI & Streamlit UI сделать полноценное приложение.